In [9]:
%load_ext autoreload
%autoreload 2

import json
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import numpy as np
from src.models.evaluate import pr_auc, precision_at_k, recall_at_precision, evaluate_all

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [2]:
# Si el score ordena perfectamente: todos los positivos arriba, todos los negativos abajo.
# PR-AUC debe ser 1.0, P@K = 1.0 mientras K <= n_positives.
y_true  = np.array([1, 1, 1, 0, 0, 0, 0, 0, 0, 0])  # 3 positivos de 10
y_score = np.array([0.9, 0.8, 0.7, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.05])

assert pr_auc(y_true, y_score) == 1.0
assert precision_at_k(y_true, y_score, 3) == 1.0
assert precision_at_k(y_true, y_score, 5) == 0.6  # 3 fraudes de 5 top scores
print("OK: caso perfecto")

OK: caso perfecto


In [3]:
# Si el score ordena al revés: todos los positivos abajo.
y_true  = np.array([1, 1, 1, 0, 0, 0, 0, 0, 0, 0])
y_score = np.array([0.1, 0.2, 0.3, 0.9, 0.8, 0.7, 0.6, 0.5, 0.4, 0.05])

assert precision_at_k(y_true, y_score, 3) == 0.0  # los top-3 son todos negativos
print(f"PR-AUC del anti-oracle: {pr_auc(y_true, y_score):.3f}")  # debería ser muy baja
print("OK: anti-oracle")

PR-AUC del anti-oracle: 0.242
OK: anti-oracle


In [4]:
# Con score random, PR-AUC ≈ base_rate (la tasa de positivos).
np.random.seed(42)
n = 10000
y_true  = (np.random.rand(n) < 0.035).astype(int)  # 3.5% base rate, como IEEE-CIS
y_score = np.random.rand(n)

base_rate = y_true.mean()
print(f"Base rate: {base_rate:.4f}")
print(f"PR-AUC random: {pr_auc(y_true, y_score):.4f}")
# Debería estar cerca de base_rate (±0.005)
assert abs(pr_auc(y_true, y_score) - base_rate) < 0.01
print("OK: random ≈ base rate")

Base rate: 0.0341
PR-AUC random: 0.0354
OK: random ≈ base rate


In [5]:
# Modelo malísimo: ningún threshold alcanza 90% de precisión.
y_true  = np.array([1, 0, 0, 0, 1, 0, 0, 0, 1, 0])
y_score = np.array([0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5])  # todos iguales

result = recall_at_precision(y_true, y_score, min_precision=0.9)
print(result)
# Como precision máxima es 0.3 (base rate), no hay threshold con P >= 0.9
assert result["threshold"] is None
assert result["recall"] == 0.0
print("OK: caso degenerado")

{'threshold': None, 'precision': None, 'recall': 0.0, 'n_flagged': 0}
OK: caso degenerado


In [7]:
y_true  = np.array([1, 1, 1, 1, 0, 0, 0, 0, 0, 0])
y_score = np.array([0.9, 0.85, 0.8, 0.4, 0.7, 0.5, 0.3, 0.2, 0.1, 0.05])

result = recall_at_precision(y_true, y_score, min_precision=0.7)
print(result)

# Inspección del threshold elegido
t = result["threshold"]
print(f"\nThreshold: {t}")
mask = y_score >= t
print(f"Flagged scores: {y_score[mask]}")
print(f"Flagged labels: {y_true[mask]}")
print(f"Precision real: {y_true[mask].mean()}")
print(f"Recall real: {y_true[mask].sum() / y_true.sum()}")

{'threshold': 0.7, 'precision': 0.75, 'recall': 0.75, 'n_flagged': 4}

Threshold: 0.7
Flagged scores: [0.9  0.85 0.8  0.7 ]
Flagged labels: [1 1 1 0]
Precision real: 0.75
Recall real: 0.75


In [10]:
np.random.seed(0)
n = 5000
y_true  = (np.random.rand(n) < 0.05).astype(int)
y_score = np.random.rand(n) * 0.3 + y_true * 0.4  # señal débil pero existe

result = evaluate_all(y_true, y_score)
print(json.dumps(result, indent=2))

# Estructura esperada
assert "pr_auc" in result
assert "precision_at_k" in result
assert set(result["precision_at_k"].keys()) == {"100", "500", "1000", "5000"}
assert set(result["recall_at_precision"].keys()) == {"0.5", "0.7", "0.9"}
assert "n_samples" in result and "n_positives" in result and "base_rate" in result
print("\nOK: evaluate_all estructura correcta")

{
  "pr_auc": 1.0,
  "precision_at_k": {
    "100": 1.0,
    "500": 0.51,
    "1000": 0.255,
    "5000": 0.051
  },
  "recall_at_precision": {
    "0.5": {
      "threshold": 0.28202512378168093,
      "precision": 0.5,
      "recall": 1.0,
      "n_flagged": 510
    },
    "0.7": {
      "threshold": 0.292423381500064,
      "precision": 0.7005494505494505,
      "recall": 1.0,
      "n_flagged": 364
    },
    "0.9": {
      "threshold": 0.29821641793942294,
      "precision": 0.901060070671378,
      "recall": 1.0,
      "n_flagged": 283
    }
  },
  "n_samples": 5000,
  "n_positives": 255,
  "base_rate": 0.051
}

OK: evaluate_all estructura correcta


In [11]:
import json
# Esto debe NO tirar TypeError (numpy floats a veces rompen json.dumps)
serialized = json.dumps(result, indent=2)
parsed = json.loads(serialized)
assert parsed == result
print("OK: JSON serializable")

OK: JSON serializable


In [ ]:
import pandas as pd
df = pd.read_parquet('/Users/feliperivas/Desktop/Code/Proyectos/fraud-detection-copilot/data/processed/train_transaction_features.parquet')
strings = df.select_dtypes(include='object').columns.tolist()
print(f"Total columnas string: {len(strings)}")
print(strings)

Total columnas string: 34
['ProductCD', 'card4', 'card6', 'P_emaildomain', 'R_emaildomain', 'M1', 'M2', 'M3', 'M4', 'M5', 'M6', 'M7', 'M8', 'M9', 'id_12', 'id_15', 'id_16', 'id_23', 'id_27', 'id_28', 'id_29', 'id_30', 'id_31', 'id_33', 'id_34', 'id_35', 'id_36', 'id_37', 'id_38', 'DeviceType', 'DeviceInfo', 'uid1', 'uid2', 'split']


/var/folders/_s/fjyv9lpx37s1b52z7j2v78gh0000gn/T/ipykernel_9444/1052629911.py:3: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  strings = df.select_dtypes(include='object').columns.tolist()
